# 01 - Preparação e qualidade dos dados (Olist)

**Objetivo:** carregar as 9 tabelas, checar a qualidade e montar uma **tabela analítica com uma linha por pedido**.

## 1. Configuração

Importa as ferramentas principais para manipulação de dados e gerenciar caminhos de pastas de forma limpa no sistema operacional.

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

## 2. Carga das tabelas

Cria um dicionário (arquivos) associando um apelido curto a cada nome de arquivo CSV.

In [2]:
arquivos = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

dfs = {nome: pd.read_csv(RAW / arq) for nome, arq in arquivos.items()}
print("Tabelas carregadas:", list(dfs))

Tabelas carregadas: ['customers', 'orders', 'items', 'payments', 'reviews', 'products', 'sellers', 'geolocation', 'category_translation']


## 3. Visão geral e qualidade por tabela

Executa uma varredura para entender o tamanho de cada base, checar se há linhas duplicadas e contar quantas células nulas (valores vazios) existem em cada uma.

In [3]:
resumo = pd.DataFrame({
    "linhas": {k: len(v) for k, v in dfs.items()},
    "colunas": {k: v.shape[1] for k, v in dfs.items()},
    "linhas_duplicadas": {k: int(v.duplicated().sum()) for k, v in dfs.items()},
    "celulas_nulas": {k: int(v.isna().sum().sum()) for k, v in dfs.items()},
})
resumo

,linhas,colunas,linhas_duplicadas,celulas_nulas
customers,99441,5,0,0
orders,99441,8,0,4908
items,112650,7,0,0
payments,103886,5,0,0
reviews,99224,7,0,145903
products,32951,9,0,2448
sellers,3095,4,0,0
geolocation,1000163,5,261831,0
category_translation,71,2,0,0


In [4]:
# Nulos por coluna (só colunas que têm nulo)
for nome, df_ in dfs.items():
    nulos = df_.isna().sum()
    nulos = nulos[nulos > 0]
    if len(nulos):
        print(f"\n--- {nome} ---")
        print((nulos / len(df_) * 100).round(2).astype(str) + " %")


--- orders ---
order_approved_at                0.16 %
order_delivered_carrier_date     1.79 %
order_delivered_customer_date    2.98 %
dtype: str

--- reviews ---
review_comment_title      88.34 %
review_comment_message     58.7 %
dtype: str

--- products ---
product_category_name         1.85 %
product_name_lenght           1.85 %
product_description_lenght    1.85 %
product_photos_qty            1.85 %
product_weight_g              0.01 %
product_length_cm             0.01 %
product_height_cm             0.01 %
product_width_cm              0.01 %
dtype: str


## 4. Chaves e granularidade

Verifica a unicidade das chaves principais antes de fazer qualquer cruzamento (join).

In [5]:
orders = dfs["orders"].copy()
customers = dfs["customers"].copy()

print("order_id único em orders?          ", orders["order_id"].is_unique)
print("customer_id único em customers?    ", customers["customer_id"].is_unique)
print("Clientes (customer_id):            ", customers["customer_id"].nunique())
print("Clientes reais (customer_unique_id):", customers["customer_unique_id"].nunique())

print("\nLinhas por pedido em cada tabela filha:")
for nome in ["items", "payments", "reviews"]:
    por_pedido = dfs[nome].groupby("order_id").size()
    print(f"  {nome:9s} -> máx {por_pedido.max()} linhas/pedido | pedidos com mais de 1: {(por_pedido > 1).sum():,}")

order_id único em orders?           True
customer_id único em customers?     True
Clientes (customer_id):             99441
Clientes reais (customer_unique_id): 96096

Linhas por pedido em cada tabela filha:
  items     -> máx 21 linhas/pedido | pedidos com mais de 1: 9,803
  payments  -> máx 29 linhas/pedido | pedidos com mais de 1: 2,961
  reviews   -> máx 3 linhas/pedido | pedidos com mais de 1: 547


## 5. Datas e status dos pedidos

Converte as colunas de texto referentes a datas (data de compra, aprovação, envio, entrega e estimativa) para o formato de data real do Pandas.

In [6]:
colunas_data = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for c in colunas_data:
    orders[c] = pd.to_datetime(orders[c])

print(orders["order_status"].value_counts())
print()
print("Nulos em datas por status (esperado: cancelados/não entregues não têm data de entrega):")
orders.groupby("order_status")[colunas_data].apply(lambda g: g.isna().sum())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Nulos em datas por status (esperado: cancelados/não entregues não têm data de entrega):


,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
order_status,,,,,
approved,0,0,2,2,0
canceled,0,141,550,619,0
created,0,5,5,5,0
delivered,0,14,2,8,0
invoiced,0,0,314,314,0
processing,0,0,301,301,0
shipped,0,0,0,1107,0
unavailable,0,0,609,609,0


## 6. Agregação ao nível do pedido

Agrega as tabelas para que cada pedido vire uma linha única com somatórios e métricas consolidadas.

In [7]:
# Itens: receita de produtos e frete por pedido
items_agg = (
    dfs["items"]
    .groupby("order_id")
    .agg(
        qtd_itens=("order_item_id", "count"),
        receita_produtos=("price", "sum"),
        frete=("freight_value", "sum"),
        qtd_sellers=("seller_id", "nunique"),
    )
    .reset_index()
)

# Pagamentos: valor total, parcelas e tipo do primeiro pagamento
pag = dfs["payments"].sort_values(["order_id", "payment_sequential"])
payments_agg = (
    pag.groupby("order_id")
    .agg(
        valor_pago=("payment_value", "sum"),
        parcelas_max=("payment_installments", "max"),
        qtd_pagamentos=("payment_sequential", "count"),
    )
    .join(pag.groupby("order_id")["payment_type"].first().rename("tipo_pagamento"))
    .reset_index()
)

# Reviews: se houver mais de uma por pedido, fica a mais recente
rev = dfs["reviews"].copy()
rev["review_creation_date"] = pd.to_datetime(rev["review_creation_date"])
reviews_last = (
    rev.sort_values("review_creation_date")
    .drop_duplicates("order_id", keep="last")[["order_id", "review_score"]]
)

print(items_agg.shape, payments_agg.shape, reviews_last.shape)

(98666, 5) (99440, 5) (98673, 2)


## 7. Tabela analítica

Faz as junções da tabela principal de orders com os dados cadastrais do cliente e com as agregações criadas no passo anterior

In [8]:
df = (
    orders
    .merge(customers[["customer_id", "customer_unique_id", "customer_city", "customer_state"]],
           on="customer_id", how="left")
    .merge(items_agg, on="order_id", how="left")
    .merge(payments_agg, on="order_id", how="left")
    .merge(reviews_last, on="order_id", how="left")
)

# Garantia anti-duplicação: o join não pode ter mudado o número de pedidos
assert len(df) == len(orders), "Join duplicou ou perdeu pedidos!"
assert df["order_id"].is_unique
print("OK:", df.shape)

OK: (99441, 20)


## 8. Métricas derivadas (logística e satisfação)

Cria novas colunas vitais para o negócio a partir das datas e notas: prazos, lead time, atrasos e satisfação.

In [9]:
DIA = 86400  # segundos

df["dias_aprovacao"] = (df["order_approved_at"] - df["order_purchase_timestamp"]).dt.total_seconds() / DIA
df["dias_postagem"] = (df["order_delivered_carrier_date"] - df["order_approved_at"]).dt.total_seconds() / DIA
df["dias_transporte"] = (df["order_delivered_customer_date"] - df["order_delivered_carrier_date"]).dt.total_seconds() / DIA
df["lead_time_total"] = (df["order_delivered_customer_date"] - df["order_purchase_timestamp"]).dt.total_seconds() / DIA

# Atraso: positivo = chegou depois do prometido
df["dias_atraso"] = (df["order_delivered_customer_date"] - df["order_estimated_delivery_date"]).dt.total_seconds() / DIA
entregue_ok = df["dias_atraso"].notna()
df["atrasado"] = (df["dias_atraso"] > 0).astype(float).where(entregue_ok)

df["nota_baixa"] = (df["review_score"] <= 2).astype(float).where(df["review_score"].notna())
df["mes_compra"] = df["order_purchase_timestamp"].dt.to_period("M")

df[["lead_time_total", "dias_atraso", "atrasado", "review_score", "nota_baixa"]].describe()

,lead_time_total,dias_atraso,atrasado,review_score,nota_baixa
count,"96,476.00","96,476.00","96,476.00","98,673.00","98,673.00"
mean,12.56,-11.18,0.08,4.09,0.15
std,9.55,10.19,0.27,1.35,0.35
min,0.53,-146.02,0.00,1.00,0.00
25%,6.77,-16.24,0.00,4.00,0.00
50%,10.22,-11.95,0.00,5.00,0.00
75%,15.72,-6.39,0.00,5.00,0.00
max,209.63,188.98,1.00,5.00,1.00


## 9. Checagens de consistência

Realiza uma conferência de auditoria para buscar inconsistências lógicas nos dados, como entregues antes da compra, pedidos sem itens ou pagamentos, e status declarados como entregues que porventura estejam sem data de entrega registrada.

In [10]:
checagens = {
    "Entrega antes da compra": (df["lead_time_total"] < 0).sum(),
    "Aprovação antes da compra": (df["dias_aprovacao"] < 0).sum(),
    "Entrega antes da postagem": (df["dias_transporte"] < 0).sum(),
    "Pedidos sem itens": df["qtd_itens"].isna().sum(),
    "Pedidos sem pagamento": df["valor_pago"].isna().sum(),
    "Pedidos sem review": df["review_score"].isna().sum(),
    "Status 'delivered' sem data de entrega": ((df["order_status"] == "delivered") & df["order_delivered_customer_date"].isna()).sum(),
}
pd.Series(checagens, name="ocorrências")

Entrega antes da compra                     0
Aprovação antes da compra                   0
Entrega antes da postagem                  23
Pedidos sem itens                         775
Pedidos sem pagamento                       1
Pedidos sem review                        768
Status 'delivered' sem data de entrega      8
Name: ocorrências, dtype: int64

In [11]:
# Flags de qualidade: o que fica de fora das análises logísticas, e por quê
df["flag_postagem_negativa"] = df["dias_postagem"] < 0          # postagem antes da aprovação
df["flag_transporte_negativo"] = df["dias_transporte"] < 0      # entrega antes da postagem
df["flag_entregue_sem_data"] = (
    (df["order_status"] == "delivered") & df["order_delivered_customer_date"].isna()
)
df["excluir_logistica"] = (
    df["flag_postagem_negativa"] | df["flag_transporte_negativo"] | df["flag_entregue_sem_data"]
)

print(f"Postagem antes da aprovação (dias_postagem < 0): {int(df['flag_postagem_negativa'].sum())}")
print(f"Entrega antes da postagem (dias_transporte < 0): {int(df['flag_transporte_negativo'].sum())}")
print(f"Status 'delivered' sem data de entrega:         {int(df['flag_entregue_sem_data'].sum())}")
print(f"Total excluído das análises logísticas:         {int(df['excluir_logistica'].sum())} de {len(df):,}")

Postagem antes da aprovação (dias_postagem < 0): 1359
Entrega antes da postagem (dias_transporte < 0): 23
Status 'delivered' sem data de entrega:         8
Total excluído das análises logísticas:         1390 de 99,441


## 10. Período confiável

Agrupa os pedidos por mês para examinar o volume de dados ao longo do tempo.

**Decisão:** o estudo usa jan/2017 a ago/2018.

**Justificativa:** antes de jan/2017 a base tem apenas 329 pedidos (set a dez/2016). Em set e out/2018 há 16 e 4 pedidos. Meses com volume tão baixo distorcem taxas mensais e comparações entre anos. Os meses fora da janela permanecem na base e podem ser incluídos se a decisão mudar.

In [12]:
JANELA_INICIO, JANELA_FIM = pd.Period("2017-01"), pd.Period("2018-08")

pedidos_mes = df.groupby("mes_compra").size().rename("pedidos").to_frame()
pedidos_mes["na_janela"] = (pedidos_mes.index >= JANELA_INICIO) & (pedidos_mes.index <= JANELA_FIM)
print(pedidos_mes)

print(f"\nPedidos dentro da janela: {pedidos_mes.loc[pedidos_mes['na_janela'], 'pedidos'].sum():,}")
print(f"Fora da janela: {pedidos_mes.loc[~pedidos_mes['na_janela'], 'pedidos'].sum():,}")

            pedidos  na_janela
mes_compra                    
2016-09           4      False
2016-10         324      False
2016-12           1      False
2017-01         800       True
2017-02        1780       True
2017-03        2682       True
2017-04        2404       True
2017-05        3700       True
2017-06        3245       True
2017-07        4026       True
2017-08        4331       True
2017-09        4285       True
2017-10        4631       True
2017-11        7544       True
2017-12        5673       True
2018-01        7269       True
2018-02        6728       True
2018-03        7211       True
2018-04        6939       True
2018-05        6873       True
2018-06        6167       True
2018-07        6292       True
2018-08        6512       True
2018-09          16      False
2018-10           4      False

Pedidos dentro da janela: 99,092
Fora da janela: 349


In [13]:
# Base para análises de entrega e satisfação: apenas pedidos entregues
entregues = df[(df["order_status"] == "delivered") & df["order_delivered_customer_date"].notna() & ~df["excluir_logistica"]].copy()
print(f"Pedidos totais: {len(df):,} | entregues com data: {len(entregues):,} ({len(entregues)/len(df):.1%})")

Pedidos totais: 99,441 | entregues com data: 95,097 (95.6%)


## 11. Salvar a base

Salva as bases tratadas.

In [14]:
df.to_csv(PROCESSED / "base_pedidos.csv", index=False)
entregues.to_csv(PROCESSED / "base_pedidos_entregues.csv", index=False)
print("Arquivos salvos em", PROCESSED.resolve())

Arquivos salvos em D:\first-tech-challenge-fiap\data\processed
